# SI-SPARK Technical Documentation
This notebook documents each script invoked by `SPARK/SI-SPARK.ipynb`. It follows the notebook order and focuses on purpose, inputs/outputs, and key methods and transformations.

## Conventions
- Paths are shown as used in the notebook; adjust if you run from a different working directory.
- Inputs/Outputs summarize the main files; many scripts also emit auxiliary QC tables and plots.
- External tool invocations are documented separately since they are not repository scripts.

## 1. Biochem Environmental Compartment Analysis
The biochem section builds a cleaned environmental feature matrix, derives PCA and EOF structure, and assigns oxygen- and GMM-based compartments with downstream comparison and transition analysis.

### merge_tables_ctd_nearest_depth.py (SPARK/biochem_modeling/merge_tables_ctd_nearest_depth.py)
- Purpose: merge geochem (table A) with CTD (table B) by nearest depth and compute Oxygen_best_available.
- Inputs: `../ref_db/new_biochem/SI_JA_Compiled_Geochem_Dec_09_Outlier_RM.csv`, `../ref_db/new_biochem/SI_JA_Compiled_CTD_Data_Dec_18_2025_Outlier_RM.csv`, `--outdir ../V4_ncbi_output/biochem_processing`.
- Outputs: `../V4_ncbi_output/biochem_processing/01_merged_nearest_depth.tsv` and `../V4_ncbi_output/biochem_processing/02_oxygen_best_available.tsv`.
- Methods/Transformations: coerce key columns (lat, lon, cruise, year, month, day, depth) and drop rows with missing keys; pre-aggregate duplicate CTD rows by mean (numeric) or first non-null (non-numeric); match each table A row to the nearest CTD depth within the same station/date group and average ties; optionally apply max depth difference; create Oxygen_best_available using CTD Oxygen where present else Table A O2.

### env_calc_density.py (SPARK/biochem_modeling/env_calc_density.py)
- Purpose: compute in-situ density (rho) and optional sigma0 from salinity, temperature, and pressure or depth.
- Inputs: `../V4_ncbi_output/biochem_processing/02_oxygen_best_available.tsv` with `--salinity-col`, `--temperature-col`, `--depth-col`, `--latitude-col`, `--longitude-col`, `--sigma0`.
- Outputs: `../V4_ncbi_output/biochem_processing/02_oxygen_best_available_density.tsv` with `density_kg_m3` and optional `sigma0_kg_m3`.
- Methods/Transformations: compute pressure from depth and latitude if pressure not provided; compute absolute salinity, conservative temperature, and density via TEOS-10 (gsw); append derived columns and write TSV.

### env_stratification_metrics.py (SPARK/biochem_modeling/env_stratification_metrics.py)
- Purpose: compute stratification metrics per profile (density, N2, MLD, PEA, pycnocline depth).
- Inputs: `../V4_ncbi_output/biochem_processing/02_oxygen_best_available_density.tsv` plus column mappings and `--profile-cols Cruise`.
- Outputs: `stratification_density_profiles.tsv`, `stratification_n2_profiles.tsv`, `stratification_summary.tsv`, `stratification_mld_timeseries.tsv`, optional `stratification_feature_traces.tsv` in `../V4_ncbi_output/biochem_processing/stratification_metrics`.
- Methods/Transformations: aggregate CTD data by depth within each profile; compute pressure, SA, CT, sigma0; compute N2 and pycnocline depth; compute MLD using density threshold (fixed or adaptive), PEA, and layer-specific metrics when layer split is enabled; compile per-profile summaries and plot diagnostics.

### Manual step (not a script)
- Purpose: remove unused columns and rename features, producing `02_oxygen_best_available_density_RJM.tsv` used by the PCA step.

### env_eigenvectors.py (SPARK/biochem_modeling/env_eigenvectors.py)
- Purpose: build a cleaned biochem feature matrix and compute PCA/EOF eigenvectors with extensive QC.
- Inputs: `../V4_ncbi_output/biochem_processing/02_oxygen_best_available_density_RJM.tsv`, `--feature-cols`, `--pc-selection`, `--anchor-depths`.
- Outputs: primary tables under `../V4_ncbi_output/env_pca/tables/` including `matrix_cleaned.csv`, `matrix_cleaned_with_sparse.csv`, `matrix_scaled.csv`, `pca_explained_variance.csv`, `pca_loadings.csv`, `eigenvectors_scores.csv`, plus PC selection and QC tables (missingness, imputation, anchor diagnostics, EOF tables, etc).
- Methods/Transformations: coerce numeric features and derive time; optional depth anchoring to common depths using observed-feature distance gating; compute missingness and mark sparse features; drop rows failing missingness thresholds; interpolate within cruise along depth with gap limits; optional log1p; standardize and run PCA; optional PC selection via parallel analysis, coverage support, and stability checks; write PCA and EOF artifacts and diagnostics.

### env_compartments_o2_soft.py (SPARK/biochem_modeling/env_compartments_o2_soft.py)
- Purpose: compute soft oxygen compartments (oxic/dysoxic/suboxic/anoxic) with logistic gates and optional sticky smoothing.
- Inputs: `../V4_ncbi_output/env_pca/tables/matrix_cleaned.csv`, oxygen column and thresholds, cruise/time/depth columns, smoothing options.
- Outputs: `o2_compartments_assignments_base.csv`, `o2_responsibilities_base.csv`, summaries and persistent/episodic labels; smoothed versions and depth scatter plots when enabled.
- Methods/Transformations: compute membership probabilities with logistic transitions around thresholds; assign compartment and uncertainty; optional forward-backward smoothing per cruise sorted by depth; label persistent vs episodic by sample fraction and span.

### env_compartments_selectk.py (SPARK/biochem_modeling/env_compartments_selectk.py)
- Purpose: choose K for GMM in PC space using metrics and stability constraints.
- Inputs: `../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv`, `../V4_ncbi_output/env_pca/tables/pc_keep_decision.csv`, K range, covariance type, optional standardization.
- Outputs: model selection metrics table and summary (BIC, ICL, entropy, stability, min cluster fraction) in `../V4_ncbi_output/env_compartments_selectk`.
- Methods/Transformations: select PC columns (keep/all/explicit), drop rows with missing PCs, optionally standardize; fit GMM for each K; compute AIC/BIC, responsibility entropy, ICL, min cluster fraction; optional CV log-likelihood and block bootstrap ARI; select K by constraint-first rule.

### env_compartments_gmm.py (SPARK/biochem_modeling/env_compartments_gmm.py)
- Purpose: fit final fixed-K GMM on PC space and produce assignments, responsibilities, and diagnostics.
- Inputs: eigenvectors scores, pc_keep decision, `--K`, optional `--matrix-cleaned` for feature-space inference.
- Outputs: `compartments_assignments_base.csv`, `compartments_assignments_smoothed.csv`, responsibility tables, compartment summaries, persistent/episodic labels, PC scatter plots, optional HDBSCAN outputs, and feature shift tables in `../V4_ncbi_output/env_compartments_gmm`.
- Methods/Transformations: select and optionally standardize PC columns; fit GMM and compute responsibilities, max_prob, entropy, knn density; optional sticky smoothing across cruises/depth; compute per-compartment summaries; optionally compute PC-feature correlations and compartment feature shifts using the cleaned matrix.

### env_hybrid_compartment_builder.py (SPARK/biochem_modeling/env_hybrid_compartment_builder.py)
- Purpose: join GMM and O2 assignments and compute cruise-level composition vectors for O2, GMM, and hybrid regimes.
- Inputs: GMM assignments and O2 assignments tables joined by `cruise_year_month_depth`.
- Outputs: cruise-level composition tables (`cruise_composition_o2.csv`, `cruise_composition_gmm.csv`, `cruise_composition_hybrid.csv`), long/topN/dominant summaries, Bray-Curtis tables, stacked bar plots, optional UMAP plots in `../V4_ncbi_output/env_hybrid_soft_compartments`.
- Methods/Transformations: inner join on a composite key; compute sample-weighted mean responsibilities per cruise; compute Bray-Curtis dissimilarities; plot stacked compositions; optional UMAP on distance matrices.

### env_stratification_anomaly_detection.py (SPARK/biochem_modeling/env_stratification_anomaly_detection.py)
- Purpose: build stratification time series and detect anomalies from a single biochem table.
- Inputs: `../V4_ncbi_output/env_pca/tables/matrix_cleaned.csv`, sample id, date/month/year, depth columns, optional feature list, optional PEA metrics.
- Outputs: `stratification_timeseries.tsv`, `annual_extremes.tsv`, `stratification_monthly_profile.pdf` in `../V4_ncbi_output/env_stratification_index`.
- Methods/Transformations: select feature columns by coverage threshold; compute distance-based stratification index with smoothing; compute z-scores; detect anomalies with IsolationForest and LocalOutlierFactor; summarize monthly profiles and annual extremes.

### eof_state_clustering.py (SPARK/biochem_modeling/eof_state_clustering.py)
- Purpose: cluster EOF scores by cruise using GMM with optional sticky smoothing over time.
- Inputs: `../V4_ncbi_output/env_pca/tables/eof_eigenvectors_scores_by_cruise.csv`, `--pcs`, `--k` or `--k auto`.
- Outputs: `cruise_states_base.tsv`, `model_selection.tsv` (if auto K), and optional `cruise_states_smoothed.tsv` in `../V4_ncbi_output/eof_states`.
- Methods/Transformations: select EOF PCs, fit GMM, compute responsibilities and max_prob; if auto K, pick best BIC across range; optional sticky smoothing along time (or blocks) for low-confidence cruises.

### eof_mode_plots.py (SPARK/biochem_modeling/eof_mode_plots.py)
- Purpose: plot EOF mode loadings as variable-by-depth heatmaps with top feature tables.
- Inputs: `../V4_ncbi_output/env_pca/tables/eof_pca_loadings.csv`, optional explained variance table.
- Outputs: per-EOF PNG/PDF heatmaps and top-feature TSVs in `../V4_ncbi_output/eof_plots`.
- Methods/Transformations: parse features named as `variable@depth`, reshape loadings into a matrix by variable and depth, plot heatmaps, and export top absolute-loading features.

### env_state_transition_analysis.py (SPARK/biochem_modeling/env_state_transition_analysis.py)
- Purpose: analyze cruise-level state transitions, persistence, and coupling for O2, GMM, and hybrid regimes.
- Inputs: `cruise_composition_o2.csv`, `cruise_composition_gmm.csv`, `cruise_composition_hybrid.csv`, optional stratification and EOF state tables.
- Outputs: transition tables (soft and dominant), persistence tables, Bray-Curtis tables, change point table, agreement summaries, coupling correlations and edges when enabled, plus plots in `../V4_ncbi_output/env_state_transitions`.
- Methods/Transformations: normalize compositions; compute soft transitions as P_t(i)*P_{t+1}(j) and dominant state sequences; compute Bray-Curtis distances and change points by threshold; compare transition agreement across regimes; optional coupling correlations with clustering and edge filtering.

### env_succession_graph.py (SPARK/biochem_modeling/env_succession_graph.py)
- Purpose: build soft succession graphs from cruise-level composition vectors.
- Inputs: cruise composition tables (O2, GMM, hybrid) and time columns.
- Outputs: succession matrices, edge lists, top successor tables, heatmaps, and network plots in `../V4_ncbi_output/env_succession_graphs`.
- Methods/Transformations: compute soft transition matrices and conditional probabilities; filter edges by top N and min probability; plot heatmaps and top-edge networks.

### env_within_gmm_hdbscan.py (SPARK/biochem_modeling/env_within_gmm_hdbscan.py)
- Purpose: run HDBSCAN within each GMM component to detect substructure.
- Inputs: `../V4_ncbi_output/env_pca/tables/eigenvectors_scores.csv` and `../V4_ncbi_output/env_compartments_gmm/tables/compartments_assignments_smoothed.csv`.
- Outputs: merged table with `subcluster` labels and probabilities plus per-component summary tables in `../V4_ncbi_output/env_compartments_gmm/within_gmm_hdbscan`.
- Methods/Transformations: merge assignments with PC coordinates; drop rows missing PC values; optionally restrict fitting to high-confidence GMM rows; run HDBSCAN per component (optional standardization); propagate labels and probabilities and record skipped components.

### env_compare_compartments.py (SPARK/biochem_modeling/env_compare_compartments.py)
- Purpose: compare O2 compartments against GMM components and assess clustering quality.
- Inputs: `matrix_cleaned_with_sparse.csv`, `eigenvectors_scores.csv`, and `compartments_assignments_smoothed.csv` plus key columns.
- Outputs: depth profiles and UMAP plots colored by O2 and GMM, confusion matrices, ARI/NMI, and clustering metric tables (silhouette, CH, DB) in `../V4_ncbi_output/env_compare_compartments`.
- Methods/Transformations: derive merge key (composite or ID), compute O2 compartments from oxygen thresholds, merge with GMM labels; compute metrics in scaled PC space and scaled biochem space; generate UMAP in biochem space; evaluate low-confidence and bootstrap diagnostics when available.

### env_compartment_feature_assoc.py (SPARK/biochem_modeling/env_compartment_feature_assoc.py)
- Purpose: quantify biochem feature associations with GMM compartments using effect sizes and bootstrapped uncertainty.
- Inputs: `matrix_cleaned_with_sparse.csv` and `compartments_assignments_smoothed.csv` with a shared sample ID.
- Outputs: per-compartment feature association tables, bootstrap confidence intervals, responsibility-feature correlation tables, and plots in `../V4_ncbi_output/env_compartment_feature_assoc`.
- Methods/Transformations: merge assignments with biochem features; compute mean/median shifts and percent shifts inside vs outside each compartment; compute effect sizes and responsibility correlations; perform cruise-block bootstrap for CIs; optional depth-adjusted associations.

### env_split_o2_by_gmm.py (SPARK/biochem_modeling/env_split_o2_by_gmm.py)
- Purpose: split O2 compartments by GMM component and optionally reassign borderline samples within each O2 class.
- Inputs: `matrix_cleaned_with_sparse.csv`, `eigenvectors_scores.csv`, `compartments_assignments_smoothed.csv`, optional UMAP embedding.
- Outputs: merged subcompartment table, counts before/after, confusion matrices, reassignment QC tables, centroids and radii, plus plots in `../V4_ncbi_output/env_o2_split_by_gmm`.
- Methods/Transformations: compute O2 compartments from thresholds; define subcompartment labels as O2 x GMM; collapse small subcompartments to `other`; compute PC-space centroids for core subcompartments; optionally reassign borderline samples within the same O2 compartment using distance-to-centroid with quantile radius; compute within-O2 silhouette metrics and generate plots.

## 2. Amplicon Sequence Variant (ASV) Pipeline
This section documents the ASV pipeline and downstream analyses as invoked in the notebook.

### run_asv_pipeline.sh (SPARK/run_asv_pipeline.sh)
- Purpose: run the Nextflow ASV pipeline with a controlled conda environment.
- Inputs: `asv_pipeline_nextflow.yml` and `SPARK/envs/controller.yml` (bootstrap env); uses `yq` to read `paths.work_dir`, `paths.output_dir`, and `paths.conda_cache_dir`.
- Outputs: pipeline outputs under the configured output directory; Nextflow work directory under `paths.work_dir`.
- Methods/Transformations: bootstraps a controller mamba env if needed; exports `NXF_WORK` and `NXF_CONDA_CACHEDIR`; runs `nextflow run` with `--params-file` and `-resume`.

### External tool: seqkit stat
- Purpose: generate read statistics for raw, fastp, filtered, and concatenated files.
- Inputs: `../V4_ncbi_input/*.fastq.gz`, `../V4_ncbi_output/fastp/*.fastq.gz`, `../V4_ncbi_output/filtered/*.fasta`, `../V4_ncbi_output/concat/concat.fasta`.
- Outputs: `../V4_ncbi_output/stats/fastq_stats.tsv`, `fastp_fastqs.tsv`, `filtered_fastqs.tsv`, `concat_fastas.tsv`.
- Methods/Transformations: uses `seqkit stat -a -T` with `nproc` threads to compute summary metrics.

### External tool: SINA alignment
- Purpose: align dereplicated ASVs to SILVA and produce alignment logs.
- Inputs: `../V4_ncbi_output/derep/derep.fasta`, SILVA reference ARB.
- Outputs: aligned fasta `derep_SINA.fasta` and log `derep_SINA.log`.
- Methods/Transformations: runs `sina` with multi-threading and log file output.

### parse_sina_log.py (SPARK/parse_sina_log.py)
- Purpose: parse SINA log files to extract alignment positions and V-region coverage.
- Inputs: `../V4_ncbi_output/derep/derep_SINA.log`.
- Outputs: `../V4_ncbi_output/derep/derep_v_regions.tsv` plus summary stats to stdout.
- Methods/Transformations: strip log prefixes; parse align_start, align_end, sequence length and quality; classify V regions using default or custom boundaries with tolerance and coverage rules; sort and write TSV.

### trim_v_sina.py (SPARK/trim_v_sina.py)
- Purpose: trim aligned sequences to specified V-region coordinates with streaming I/O.
- Inputs: `../V4_ncbi_output/derep/derep_v_regions.tsv`, `../V4_ncbi_output/derep/derep_SINA.fasta`, regions `V4`, output `../V4_ncbi_output/derep/derep_trim_V4.fasta`.
- Outputs: trimmed FASTA (degapped unless `--keep-gaps`).
- Methods/Transformations: filter metadata to target regions; determine trim coordinate span; trim aligned sequences using gap-aware coordinate mapping; remove gaps and convert U to T; process in threaded batches for memory control.

### qiime_vs_classifier.py (SPARK/qiime_vs_classifier.py)
- Purpose: chunked QIIME2 VSEARCH classifier with checkpointing and per-chunk results.
- Inputs: `../V4_ncbi_output/ASVs/ASVs.upper.fasta`, SILVA taxonomy and sequence QZA files.
- Outputs: `../V4_ncbi_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv` and `ASV_SILVA_stats.full-length.vsearch.tsv` plus temporary artifacts under `intermediate/`.
- Methods/Transformations: split FASTA into chunks; import each chunk as QIIME2 artifact; run `classify_consensus_vsearch`; save per-chunk TSVs; concatenate into final TSV; compute taxonomy counts; remove intermediates.

### External tool: awk toupper
- Purpose: normalize FASTA sequences to uppercase for QIIME2 classification.
- Inputs: `../V4_ncbi_output/ASVs/ASVs_filtered.fasta`.
- Outputs: `../V4_ncbi_output/ASVs/ASVs.upper.fasta`.

### asv_pipeline.nf (SPARK/asv_pipeline.nf)
- Purpose: Nextflow DSL2 pipeline that orchestrates the ASV workflow and optional reporting steps.
- Config handling: uses `--config` if provided, else inline params, else `SPARK/asv_pipeline_nextflow.yml`; resolves paths relative to the config root; uses `paths.manifest` if provided, otherwise scans `paths.input_dir` and pairs reads by `filename_patterns` tokens; per-sample thread count is `resources.threads`; creates output subdirs for fastp/merged/filtered/concat/derep/sina/denoise/nochimeras/ASVs/mito/taxonomy/stats/logs.
- Reference handling: downloads the SILVA SINA `.arb` reference and QIIME2 SILVA taxonomy/sequence `.qza` files into output subdirectories when not found at configured paths.
- Primary flow (always run):
  - FASTP_QC: runs `fastp` trimming -> `fastp/*.fastq.gz` plus per-sample `fastp.json/html`.
  - MERGE_READS: `vsearch --fastq_mergepairs` (or pass-through for single-end) -> `merged/*.merged.fastq`.
  - FILTER_READS: `vsearch --fastx_filter` by maxEE/length -> `filtered/*.filtered.fasta`.
  - CONCAT + DEREPLICATE: concatenate filtered FASTA and `vsearch --derep_fulllength` -> `derep/derep.fasta`.
  - SINA_TRIM: `sina` alignment + `parse_sina_log.py` + `trim_v_sina.py` -> `sina/derep_trimmed.fasta`, `sina/derep_SINA.log`, `sina/derep_v_regions.tsv`.
  - DENOISE: `vsearch --cluster_unoise` -> `denoise/centroids.fasta`.
  - CHIMERA_CHECK: `vsearch --uchime3_denovo` -> `nochimeras/nochimeras.fasta`.
  - CREATE_COUNT_MATRIX: `vsearch --usearch_global` (id 0.999) -> `ASVs/ASV_counts.tsv`, `ASVs/ASVs.fasta`.
  - FILTER_TABLE: `filter_ASV_table.py` thresholds -> `ASVs/ASV_filtered.tsv`, `ASVs/ASVs_filtered.fasta`.
  - TAXONOMY: uppercase FASTA + `qiime_vs_classifier.py` -> `taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv` and stats TSV.
- Optional branches (enabled in YAML):
  - MITOMASTER + MITO_DECONTAM: chunk ASVs, MITOMASTER API, BLAST mito/contaminants, `mito_checker.py` -> `mito/mitomap/nontarget.master.tsv` and summaries.
  - FILTER_COUNTS: `filter_nontarget.py` to produce microbial/mito filtered tables.
  - GENERAL_STATS: `seqkit stat` on raw/fastp/filtered/concat.
  - PLOT_METADATA: `plot_metadata.py` to produce `metadata_updated_*`, `ASV_meta_*`, and `ASV_final.*.tsv`.
  - ASV_BATCH_CORRECTION: `asv_batch_correction.py` -> CLR tables + UMAP cluster results.
  - OUTLIER_CHECKER: `outlier_checker.py` on CLR results.
  - COLLECTORS_CURVE: `collectors_curve.py`.
  - SANKEY: `sankey_builder.py`.


### mitomaster.py (SPARK/mitomaster.py)
- Purpose: submit FASTA chunks to MITOMASTER and concatenate results with safe resume.
- Inputs: `../V4_ncbi_output/ASVs/chunks/*.fasta` and `--output-file ../V4_ncbi_output/mito/mitomap/mitomaster_output.tsv`.
- Outputs: combined MITOMASTER TSV and a checkpoint file (`.done`) in the output directory.
- Methods/Transformations: POST each FASTA to the MITOMASTER endpoint with retry and timeout; append responses with header control; use thread pool concurrency; optionally respect checkpoint to skip processed chunks.

### External tool: blastn
- Purpose: screen ASVs against mitochondrial and contaminant databases.
- Inputs: `../V4_ncbi_output/ASVs/ASVs_filtered.fasta` with `../ref_db/mito_ncbi` and `../ref_db/ssu_pipeline_contaminants`.
- Outputs: `mito_ncbi.blast6.tsv` and `ssu_pipeline_contaminants.blast6.tsv` in `../V4_ncbi_output/mito/mitomap`.
- Methods/Transformations: BLAST6 output with identity, alignment length, and evalue fields for downstream filtering.

### mito_checker.py (SPARK/mito_checker.py)
- Purpose: integrate MITOMASTER, BLAST, and SILVA taxonomy to classify non-target ASVs and summarize counts.
- Inputs: MITOMASTER TSV, mito BLAST6, SILVA taxonomy TSV, BioFactorial BLAST6.
- Outputs: `nontarget.master.tsv` plus cumulative and per-step summary TSVs and plots in `../V4_ncbi_output/mito/mitomap`.
- Methods/Transformations: filter BLAST hits by percent identity and coverage; flag each ASV by BioFactorial, NB mitochondrial substring, MITOMASTER, and BLAST mito; build master table and cumulative/non-cumulative summaries; plot line and bar charts.

### filter_nontarget.py (SPARK/filter_nontarget.py)
- Purpose: remove non-target ASVs and apply abundance and taxonomy filters.
- Inputs: `ASV_filtered.tsv`, `nontarget.master.tsv`, SILVA taxonomy TSV, metadata with group column and sample ID column.
- Outputs: `ASV_target.tsv` plus intermediate `ASV_target.decon.tsv`, `ASV_target.micro.tsv`, and `ASV_target.mito.tsv` (when `--save-intermediates`) under `../V4_ncbi_output`.
- Methods/Transformations: normalize ASV IDs; filter samples by group size; filter by BioFactorial and mitochondrial flags; separate microbial vs mitochondrial ASVs; apply relative abundance threshold and taxonomy quality filter; write final and intermediate tables.

### sankey_builder.py (SPARK/sankey_builder.py)
- Purpose: build Sankey diagrams for read/ASV flow by sample group.
- Inputs: metadata, sample manifest, fastq stats, filtered stats, ASV raw/decon/micro tables.
- Outputs: labeled and unlabeled HTML Sankey files under `../V4_ncbi_output/metadata`.
- Methods/Transformations: map FASTQ files to sample IDs; collapse read counts by sample and group; compute step totals and group-specific input/output; render Plotly Sankey with optional loss nodes.

### plot_metadata.py (SPARK/plot_metadata.py)
- Purpose: build ASV master tables and plots for microbial and mitochondrial subsets.
- Inputs: metadata, sample manifest, fastq stats, taxonomy TSV, ASV_target.micro.tsv and ASV_target.mito.tsv.
- Outputs: `metadata/ASV_meta_micro.tsv`, `metadata/master_table_micro.tsv`, `metadata/metadata_updated_micro.tsv`, `ASVs/ASV_final.micro.tsv`, plus analogous mito outputs and multiple plots (swarmplots, clustermaps, violin plots).
- Methods/Transformations: align sample IDs via manifest; convert ASV matrices to long form and merge taxonomy; optional include-rank filters; compute per-sample read totals; subtract control counts (scope and skin) to generate corrected counts; produce final ASV matrices; compute shared-ASV percent matrix and clustermaps; generate violin plots of shared ASVs.

### asv_batch_correction.py (SPARK/asv_batch_correction.py)
- Purpose: correct batch effects in CLR-transformed ASV data with before/after diagnostics.
- Inputs: ASV table, metadata with batch column, optional biological covariates.
- Outputs: `asv_clr_before_correction.tsv`, `asv_clr_after_correction.tsv`, `asv_clr_after_correction_with_metadata.tsv`, `umap_hdbscan_results.tsv`, optimization tables, and diagnostic plots in `../V4_ncbi_output/batch_correction`.
- Methods/Transformations: align samples and filter zero features; apply multiplicative replacement and CLR; apply ComBat-style correction (pycombat if available, mean-shift fallback); compute UMAP and HDBSCAN before/after; optional joint parameter optimization; generate UMAP comparison and swarm plots; compute per-feature batch effect statistics.

### assign_compartments.py (SPARK/assign_compartments.py)
- Purpose: depth stratification and compartment analysis (Figure 1 style) integrating ASV and biochem signals.
- Inputs: batch-corrected CLR table, ASV counts, ASV FASTA, metadata, depth/month columns, biochem feature list.
- Outputs: Figure panels (dendrogram heatmap, UMAP by depth/cluster, richness, between-depth distances), tables `depth_clusters.tsv`, `between_depth_distances.tsv`, `asv_richness.tsv`, `compartment_umap_clusters.tsv`, and summary text under `../V4_ncbi_output/compartments`.
- Methods/Transformations: compute GC content per ASV and sample-weighted GC; scale biochem features; compute richness; compute distance matrices and hierarchical clustering; choose cluster count; compute UMAP and assign clusters; plot depth profiles and time sections; export summary metrics.

### trajectory_analysis.py (SPARK/trajectory_analysis.py)
- Purpose: seasonal trajectory analysis comparing depth groups and data-driven clusters.
- Inputs: UMAP cluster table, CLR ASV data, metadata with month and group columns.
- Outputs: trajectory plots, distance and correlation summaries in `../V4_ncbi_output/trajectory_analysis`.
- Methods/Transformations: standardize month ordering, aggregate by group and month with years as replicates; compute trajectory distances and directionality; compute top taxa contributions and group correlations.

### aligned_biochem_visuals.py (SPARK/aligned_biochem_visuals.py)
- Purpose: cruise-sliced UMAP on biochem features with depth anchors and interactive trajectory visualization.
- Inputs: biochem metadata table with required features, depth and oxygen columns, anchoring and imputation settings.
- Outputs: UMAP plots, transition entropy tables and plots, and interactive HTML with depth/compartment controls in `../V4_ncbi_output/biochem_alignment`.
- Methods/Transformations: build depth anchors by rounded depths and coverage; impute within cruises; run UMAP; compute O2 compartment labels; optionally run HDBSCAN in UMAP and feature space; compute transition entropy per depth and render interactive plots.

### outlier_checker.py (SPARK/outlier_checker.py)
- Purpose: ensemble outlier detection for ASV profiles with per-group modeling.
- Inputs: ASV table (counts or CLR), metadata with group columns.
- Outputs: outlier tables and summary files in `../V4_ncbi_output/outliers_corrected`.
- Methods/Transformations: align samples; optional CLR; standardize features; run IsolationForest, OneClassSVM, and HDBSCAN per group; combine votes into consensus outliers; export results.

### collectors_curve.py (SPARK/collectors_curve.py)
- Purpose: species-accumulation (collector) curves by group with permutation envelopes.
- Inputs: ASV counts table and metadata with group column.
- Outputs: overlay and faceted plots plus per-group stats tables in `../V4_ncbi_output/metadata`.
- Methods/Transformations: randomize sample order per group across permutations; compute unique ASVs after k samples; compute percentile envelopes; plot curves with sample count markers.

### plot_upset.py (SPARK/plot_upset.py)
- Purpose: ASV overlap analysis with UpSet and optional Venn plots.
- Inputs: ASV raw/final tables, metadata with group and color columns, taxonomy table.
- Outputs: UpSet plots (unique and weighted), optional Venn plots, presence and sum tables in `../V4_ncbi_output/metadata`.
- Methods/Transformations: melt counts to long form; build group sets and per-group totals; generate UpSet plots with stacked bars; compute exclusive membership tables and sums.

### bubbleplotter.py (SPARK/bubbleplotter.py)
- Purpose: taxonomy bubble plots of ASV counts across depths and months.
- Inputs: `../V4_ncbi_output/metadata/ASV_meta_micro.tsv` with taxonomy and counts.
- Outputs: per-depth bubble plots and summary bubble plots in `../V4_ncbi_output/metadata`.
- Methods/Transformations: normalize taxonomy strings and fill unclassified ranks; aggregate counts by sample and taxonomy; transform counts for bubble sizes; order taxonomy with spacing; draw hierarchy labels and summaries.

### umap_clustering.py (SPARK/umap_clustering.py)
- Purpose: UMAP + HDBSCAN clustering on ASV count data with metadata overlays.
- Inputs: `../V4_ncbi_output/metadata/ASV_meta_micro.tsv` and metadata columns for depth/month.
- Outputs: UMAP plots and clustering tables in `../V4_ncbi_output/metadata`.
- Methods/Transformations: pivot long table to sample x ASV matrix; normalize and transform counts; standardize; run UMAP and HDBSCAN; merge metadata and generate plots.

### calc_div.py (SPARK/calc_div.py)
- Purpose: compute alpha (Shannon) and beta (Bray-Curtis, Jaccard) diversity matrices.
- Inputs: `../V4_ncbi_output/ASVs/ASV_final.micro.tsv` with samples as columns.
- Outputs: `../V4_ncbi_output/diversity/shannon.tsv`, `bray.tsv`, `jaccard.tsv`.
- Methods/Transformations: load counts, orient samples as rows, drop empty rows/cols, compute Shannon with skbio, compute pairwise Bray-Curtis and Jaccard (presence threshold optional).

### plot_diversity.py (SPARK/plot_diversity.py)
- Purpose: alpha and beta diversity analytics with PERMANOVA and UMAP visualizations.
- Inputs: metadata, alpha table, Bray and Jaccard distance matrices.
- Outputs: alpha t-tests, alpha boxplots, PERMANOVA tables and heatmaps, UMAP coordinate tables and plots in `../V4_ncbi_output/diversity`.
- Methods/Transformations: merge alpha metrics with metadata; run pairwise t-tests with FDR; run global and pairwise PERMANOVA; compute UMAP from distance matrices; plot UMAP colored by group and optional secondary columns; supports mito mode.

### run_indicspecies.R (SPARK/run_indicspecies.R)
- Purpose: run indicspecies multipatt for indicator ASVs by group.
- Inputs: ASV count table, metadata, sample column, group columns.
- Outputs: `indicspecies/*_indicator_species_results.tsv` and `*_indicator_species_summary.tsv` under `../V4_ncbi_output/indicspecies`.
- Methods/Transformations: align ASV counts with metadata; drop groups with small sample size; run `multipatt` with duleg off and on; compute q-values; write results.

### plot_indicspecies.py (SPARK/plot_indicspecies.py)
- Purpose: build enriched ISA tables and publication-grade scatter plots.
- Inputs: indicspecies results for two groupings, optional Venn presence table and taxonomy table, label and palette mappings.
- Outputs: enriched ISA tables and plots (type, status, combined, optional phylum-colored) in `../V4_ncbi_output/indicspecies`.
- Methods/Transformations: map index values to labels and colors; compute -log10 p and significance; jitter points to avoid overlap; merge group1 and group2 results; optionally overlay Venn labels and taxonomy.

### plot_clustermaps.py (SPARK/plot_clustermaps.py)
- Purpose: log-scaled abundance clustermaps across ranks and ASVs with sample color bars.
- Inputs: `ASV_meta_micro.tsv`, `metadata_updated_micro.tsv`, optional ISA results, optional mito ASV table, palettes and type order.
- Outputs: clustermap plots (code and clustered) and pivot tables per rank in `../V4_ncbi_output/diversity` and optional mito outputs.
- Methods/Transformations: select taxa per rank by top N within type_group plus ISA ASVs; map non-selected taxa to `Other`; pivot to taxa x sample_code; log10 transform counts; draw clustermaps with column color bars.

### run_spieceasi.R (SPARK/run_spieceasi.R)
- Purpose: infer microbial association networks using SpiecEasi with caching and graph exports.
- Inputs: `../V4_ncbi_output/ASVs/ASV_final.micro.tsv` and filtering/SpiecEasi parameters.
- Outputs: filtered counts RDS, SpiecEasi RDS, precision and partial correlation matrices, GraphML/GML networks, adjacency matrices, edge list, node features, and layout PDFs under `../V4_ncbi_output/spieceasi`.
- Methods/Transformations: optionally transpose and filter ASVs by abundance and prevalence; run SpiecEasi with pulsar; compute precision and partial correlations; threshold positive edges; compute graph layouts; export networks and node centrality tables.

### graph_network.py (SPARK/graph_network.py)
- Purpose: visualize SpiecEasi GraphML networks with ISA, Venn, and taxonomy overlays.
- Inputs: GraphML files, node_features.csv, ASV counts, taxonomy table, Venn presence table, indicspecies summaries.
- Outputs: network plots by degree, abundance, ISA type/status, Venn, and phylum in `../V4_ncbi_output/spieceasi`.
- Methods/Transformations: load graphs and compute mean abundance; merge ISA summaries and taxonomy into node attributes; compute or load cached spring layouts; draw multiple network modes with legend scaling and optional labels.